In [2]:
import math
import time
import random
import torch
from datasets import load_dataset
import _referAsMain
import sys
import numpy
print(sys.version_info)

/home/holo/dev/PFE_LLM_art_generation/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


added '/home/holo/dev/PFE_LLM_art_generation' to import paths
sys.version_info(major=3, minor=10, micro=19, releaselevel='final', serial=0)


In [3]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

Torch version: 2.9.1+cu128
CUDA available: True
CUDA version: 12.8
Device count: 1


In [4]:
import LLM.nanochat.gpt as nanoChatModel
import LLM.nanochat.tokenizer as tokenizerLib
from LLM.nanochat.common import compute_init, autodetect_device_type

from LLM import Model
from metrics.metrics import get_learning_rate

In [5]:
device_type = "cuda"
device_type = (autodetect_device_type() if device_type == "" else device_type)
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = \
    compute_init(device_type)

2026-03-04 11:01:23,961 - LLM.nanochat.common - INFO - Distributed world size: 1


In [46]:
aspect_ratio = 10.5
head_dim = 128
max_seq_len = 2048*8
window_pattern = "SSSL"
vocab_size = 2048


def build_model_meta(depth):
    """Build a model on meta device for a given depth (shapes/dtypes only, no data)."""
    # Model dim is nudged up to nearest multiple of head_dim for clean division
    # (FA3 requires head_dim divisible by 8, and this guarantees head_dim == args.head_dim exactly)
    base_dim = depth * aspect_ratio
    model_dim = int(((base_dim + head_dim - 1) // head_dim) * head_dim)
    num_heads = int(model_dim // head_dim)
    print(f"num heads: {model_dim / head_dim}")
    config = nanoChatModel.GPTConfig(
        sequence_len=max_seq_len, vocab_size=vocab_size,
        n_layer=depth, n_head=num_heads, n_kv_head=num_heads, n_embd=model_dim,
        window_pattern=window_pattern)
    print(config)
    with torch.device("meta"):
        model_meta = nanoChatModel.GPT(config)
    return model_meta

model = Model()

model.llm = build_model_meta(6)
# 2) All tensors get storage on target device but with uninitialized (garbage) data
model.llm.to_empty(device=device)
model.llm.init_weights()  # 3) All tensors get initialized
model.optimizer = model.llm.setup_optimizer()

params = model.llm.num_scaling_params()
print(params.items())
params_Embed = (params['wte'] + params['value_embeds'])
print(f"{params['total']:_d} params "
      f"(with embeding: {params_Embed:_d} | "
      f"last layer: {params['lm_head']:_d} | "
      f"transformer: {params['transformer_matrices']:_d})")
model.llm = model.llm.bfloat16()

# the inputs to model will never change shape so dynamic=False is safe
model.llm = torch.compile(model.llm, dynamic=False) # type: ignore
# model.eval();


num heads: 1.0
GPTConfig(sequence_len=16384, vocab_size=2048, n_layer=6, n_head=1, n_kv_head=1, n_embd=128, window_pattern='SSSL')
Scaling the LR for the AdamW parameters ∝1/√(128/768) = 2.449490
dict_items([('wte', 262144), ('value_embeds', 786432), ('lm_head', 262144), ('transformer_matrices', 1179744), ('scalars', 12), ('total', 2490476)])
2_490_476 params (with embeding: 1_048_576 | last layer: 262_144 | transformer: 1_179_744)


In [47]:
get_learning_rate(model)

0.009797958971132713

In [48]:
lst = ["lm_head", "embedding",  "value_embeds", "residuals", "x0", ] + ["transformers"] * 4
for i, optim in enumerate(model.optimizer.param_groups):
    name = lst[i]
    nbParams = 0
    for paramGrp in optim['params']:
        nbParams += numpy.prod(paramGrp.shape)
    print(f"lr of {i}[{optim['kind']}, {name}, with {nbParams} params, {paramGrp.shape}] is {optim['lr']}")

lr of 0[adamw, lm_head, with 262144 params, torch.Size([2048, 128])] is 0.009797958971132713
lr of 1[adamw, embedding, with 262144 params, torch.Size([2048, 128])] is 0.4898979485566357
lr of 2[adamw, value_embeds, with 786432 params, torch.Size([2048, 128])] is 0.4898979485566357
lr of 3[adamw, residuals, with 6 params, torch.Size([6])] is 0.005
lr of 4[adamw, x0, with 6 params, torch.Size([6])] is 0.5
lr of 5[muon, transformers, with 96 params, torch.Size([1, 32])] is 0.02
lr of 6[muon, transformers, with 393216 params, torch.Size([128, 128])] is 0.02
lr of 7[muon, transformers, with 393216 params, torch.Size([128, 512])] is 0.02
lr of 8[muon, transformers, with 393216 params, torch.Size([512, 128])] is 0.02
